# 🏥 PuskesmasAI — Offline-First AI Triage Assistant for Rural Indonesia

> **Gemma 4 Good Hackathon** · Kaggle × Google DeepMind  
> **Track:** Main Track + Ollama (Special Technology)  
> **Model:** Gemma 4 E2B via Ollama (local) · Gemini API (Kaggle demo fallback)  
> **Author:** Jefri · Solo Founder · Makassar, South Sulawesi, Indonesia

---

## ⚠️ Note to Judges

**In production** (local deployment), PuskesmasAI runs `gemma4:e2b` via Ollama:
```bash
ollama pull gemma4:e2b
ollama run gemma4:e2b
# Verified working: Intel i5-1235U · 8GB RAM · Windows 11 · fully offline ✅
```

**In this Kaggle notebook**, since Ollama cannot run in cloud environments, we use the **Gemini API (Gemma family)** as a compatible fallback. The prompt engineering, JSON schema, safety layers, and triage logic are **exactly the same** in both environments.

---
## 🌍 Problem Statement

Indonesia has **1 doctor per 5,000 people** — far below WHO's recommended 1:600 ratio.  
In 3T regions (Tertinggal, Terdepan, Terluar), over **45% of Puskesmas** lack adequate medical staff.  
**1.5 million kaders** must make triage decisions without internet, without doctors nearby.

**PuskesmasAI** = PWA + Gemma 4 E2B via Ollama. Fully offline. Works on low-end hardware.

In [ ]:
!pip install requests -q

import json, requests, os
from datetime import datetime
from IPython.display import display, HTML

# ── CONFIG ──────────────────────────────────────────────
# Set True when running locally with Ollama
OLLAMA_AVAILABLE = False

OLLAMA_URL   = "http://localhost:11434/api/chat"
OLLAMA_MODEL = "gemma4:e2b"

GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY", "")
GEMINI_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemma-3-4b-it:generateContent?key={GEMINI_API_KEY}"

print("✅ PuskesmasAI ready")
print(f"   Runtime : {'Ollama — gemma4:e2b (offline)' if OLLAMA_AVAILABLE else 'Gemini API — Kaggle demo fallback'}")

---
## 🔐 Security — Prompt Injection Guard

In [ ]:
BLOCKED_PATTERNS = ["ignore previous instructions","you are now","system:","assistant:","<|im_start|>","forget everything","disregard"]

def sanitize_input(text: str) -> str:
    for pattern in BLOCKED_PATTERNS:
        if pattern.lower() in text.lower():
            print(f"⚠️  Injection blocked: '{pattern}'")
            return "[SANITIZED INPUT]"
    return text[:500]

print("✅ Security layer active")

---
## 🧠 Prompt Engineering — Structured Clinical Triage

In [ ]:
SYSTEM_PROMPT = """Anda adalah asisten triage klinis untuk kader kesehatan masyarakat Indonesia.
Nilai gejala pasien dan berikan rekomendasi triage terstruktur.
SELALU jawab HANYA dengan JSON valid. Tidak ada teks di luar JSON.

Level triage:
- GREEN  : Tidak mendesak — tangani di rumah
- YELLOW : Perlu dipantau — rujuk ke Puskesmas dalam 24 jam  
- ORANGE : Mendesak — rujuk ke Puskesmas segera
- RED    : Darurat — panggil ambulans sekarang"""

def build_prompt(age, sex, symptoms, temp):
    symptoms = sanitize_input(symptoms)
    return f"""Data pasien: usia {age} tahun, {sex}, suhu {temp}°C.
Keluhan: {symptoms}

Jawab dengan JSON:
{{
  "triage_level": "GREEN|YELLOW|ORANGE|RED",
  "recommendation": "tindakan jelas untuk kader",
  "possible_conditions": ["kondisi1", "kondisi2"],
  "immediate_actions": ["tindakan1", "tindakan2", "tindakan3"],
  "red_flags": ["tanda bahaya1", "tanda bahaya2"],
  "confidence": 0.0,
  "disclaimer": "Alat bantu keputusan — bukan pengganti dokter."
}}"""

print("✅ Prompt builder ready")

---
## 🔌 AI Client — Ollama local / Gemini API fallback

In [ ]:
def call_ollama(age, sex, symptoms, temp):
    """PRIMARY: gemma4:e2b via Ollama — verified on i5-1235U, 8GB RAM"""
    r = requests.post(OLLAMA_URL, json={
        "model": OLLAMA_MODEL,
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": build_prompt(age, sex, symptoms, temp)}
        ],
        "stream": False, "format": "json", "options": {"temperature": 0.1}
    }, timeout=60)
    return json.loads(r.json()["message"]["content"])

def call_gemini(age, sex, symptoms, temp):
    """FALLBACK: Gemini API — identical prompt & schema"""
    prompt = SYSTEM_PROMPT + "\n\n" + build_prompt(age, sex, symptoms, temp)
    r = requests.post(GEMINI_URL, json={
        "contents": [{"parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": 0.1, "responseMimeType": "application/json"}
    }, timeout=30)
    text = r.json()["candidates"][0]["content"]["parts"][0]["text"]
    return json.loads(text.replace("```json","").replace("```","").strip())

def run_triage(age, sex, symptoms, temp="tidak tercatat"):
    try:
        if OLLAMA_AVAILABLE:
            print("🖥️  Ollama gemma4:e2b — offline mode")
            return call_ollama(age, sex, symptoms, temp)
        else:
            print("☁️  Gemini API — Kaggle demo fallback")
            return call_gemini(age, sex, symptoms, temp)
    except Exception as e:
        return {"error": str(e)}

print("✅ AI client ready")

---
## 🎨 Color-coded Result Display

In [ ]:
TRIAGE_CONFIG = {
    "GREEN":  {"emoji":"🟢","color":"#16a34a","bg":"#dcfce7","label":"TIDAK MENDESAK"},
    "YELLOW": {"emoji":"🟡","color":"#ca8a04","bg":"#fef9c3","label":"PERLU DIPANTAU"},
    "ORANGE": {"emoji":"🟠","color":"#ea580c","bg":"#ffedd5","label":"MENDESAK"},
    "RED":    {"emoji":"🔴","color":"#dc2626","bg":"#fee2e2","label":"DARURAT"},
}

def show(result, patient):
    if "error" in result:
        print(f"❌ {result['error']}"); return
    level = result.get("triage_level","UNKNOWN")
    c = TRIAGE_CONFIG.get(level, {"emoji":"⚪","color":"#6b7280","bg":"#f3f4f6","label":level})
    conf = int(result.get("confidence",0)*100)
    cond = "".join(f"<li>{x}</li>" for x in result.get("possible_conditions",[]))
    acts = "".join(f"<li>{x}</li>" for x in result.get("immediate_actions",[]))
    flags= "".join(f"<li style='color:#dc2626'>⚠️ {x}</li>" for x in result.get("red_flags",[]))
    display(HTML(f"""
    <div style="font-family:sans-serif;max-width:640px;border-radius:12px;overflow:hidden;border:2px solid {c['color']};margin:12px 0">
      <div style="background:{c['color']};color:white;padding:16px 20px">
        <h2 style="margin:0">{c['emoji']} {level} — {c['label']}</h2>
        <p style="margin:4px 0 0;font-size:13px;opacity:.9">{patient['age']}th · {patient['sex']} · {patient.get('temp','?')}°C · Confidence: {conf}%</p>
      </div>
      <div style="background:{c['bg']};padding:16px 20px">
        <p style="font-weight:bold;color:{c['color']}">{result.get('recommendation','')}</p>
        <b>🩺 Kemungkinan:</b><ul>{cond}</ul>
        <b>⚡ Tindakan:</b><ul>{acts}</ul>
        {'<b>🚨 Red Flags:</b><ul>'+flags+'</ul>' if flags else ''}
        <p style="font-size:11px;color:#6b7280;margin-top:12px">{result.get('disclaimer','')}</p>
      </div>
    </div>"""))

print("✅ Display ready")

---
## 🧪 Case 1 — Suspek Dengue · Expected: ORANGE

In [ ]:
p1 = {"age":32,"sex":"perempuan","temp":"38.8",
      "symptoms":"demam 3 hari, bintik merah di kulit, mual, tidak nafsu makan, nyeri sendi"}
print(f"👤 {p1['age']}th {p1['sex']} · {p1['symptoms']}\n")
r1 = run_triage(**p1)
show(r1, p1)
print(json.dumps(r1, indent=2, ensure_ascii=False))

---
## 🧪 Case 2 — Demam Ringan Anak · Expected: GREEN

In [ ]:
p2 = {"age":8,"sex":"laki-laki","temp":"37.6",
      "symptoms":"demam ringan sejak kemarin, pilek, batuk sedikit, masih aktif bermain, nafsu makan normal"}
print(f"👤 {p2['age']}th {p2['sex']} · {p2['symptoms']}\n")
r2 = run_triage(**p2)
show(r2, p2)
print(json.dumps(r2, indent=2, ensure_ascii=False))

---
## 🧪 Case 3 — Suspek Stroke · Expected: RED

In [ ]:
p3 = {"age":65,"sex":"laki-laki","temp":"36.5",
      "symptoms":"tiba-tiba wajah miring sebelah, tangan kiri lemas, bicara pelo, terjadi 30 menit lalu"}
print(f"👤 {p3['age']}th {p3['sex']} · {p3['symptoms']}\n")
r3 = run_triage(**p3)
show(r3, p3)
print(json.dumps(r3, indent=2, ensure_ascii=False))

---
## 💾 Offline Patient Log — Simulasi IndexedDB

In [ ]:
log = []
EMOJI = {"GREEN":"🟢","YELLOW":"🟡","ORANGE":"🟠","RED":"🔴"}

for p, r in [(p1,r1),(p2,r2),(p3,r3)]:
    log.append({"id":len(log)+1,"timestamp":datetime.now().isoformat()[:19],
                "age":p["age"],"sex":p["sex"],
                "triage_level":r.get("triage_level","?"),
                "recommendation":r.get("recommendation",""),"synced":False})

print("📋 PATIENT LOG (IndexedDB — stored offline on kader device)")
print("="*65)
for rec in log:
    e = EMOJI.get(rec['triage_level'],'⚪')
    print(f"{e} [{rec['timestamp']}] #{rec['id']} · {rec['age']}th {rec['sex']} → {rec['triage_level']}")
    print(f"   {rec['recommendation']}")
    print(f"   Sync: {'✅' if rec['synced'] else '⏳ pending (offline)'}\n")

---
## 🏗️ Architecture
```
┌─────────────────────────────────────────────┐
│         LOCAL SERVER / LAPTOP               │
│  ┌──────────────┐   ┌────────────────────┐  │
│  │ PWA (Chrome) │──▶│ Ollama gemma4:e2b  │  │
│  │ Next.js 14   │   │ localhost:11434    │  │
│  │ IndexedDB    │   │ i5+8GB RAM ✅      │  │
│  └──────┬───────┘   └────────────────────┘  │
└─────────┼───────────────────────────────────┘
          │ (online only)
          ▼
   ┌─────────────┐
   │ Turso libSQL│ ← sync log (no symptoms)
   └─────────────┘
```
✅ Offline-first · ✅ Zero cloud dependency · ✅ No patient data leaves device

---
## 📊 Impact
| Metrik | Target |
|---|---|
| Target pengguna | ~1.5 juta kader |
| Populasi terjangkau | ~60 juta (wilayah 3T) |
| Internet yang dibutuhkan | Nol setelah setup |
| Data pasien ke cloud | Tidak pernah |
| Hardware minimum | i5 + 8GB RAM |

> *"AI terbaik adalah yang bekerja untuk mereka yang paling membutuhkan — bahkan saat internet tidak ada."*

**Author:** Jefri · Makassar, Indonesia · **License:** MIT